# 03 · Demand Forecasting

**The constraint that drives everything:** 74.9% of SKU-days have zero demand.

### Why not Prophet?

Not because it is bad — because it is wrong for *this* data. Prophet models a
smooth trend plus smooth seasonalities. Fitted to a series that is zero four days
in five, it produces smooth non-negative drift that is neither accurate nor
interpretable, at ~1–2 s per SKU (40–80 min per backtest fold set). Popularity is
not a modelling argument.

### What was compared (10 models)

| Family | Models |
|---|---|
| Baselines | naive, seasonal naive (7d), moving average (28d) |
| Annual seasonality | annual naive (364d), annual scaled, **seasonal index** |
| Intermittent demand | Croston, Croston-SBA |
| Statistical | ETS / Holt-Winters, SARIMA |

### Validation

Rolling-origin backtest, **5 folds × 30 days**. Fold *k* trains only on
`demand[:origin]` — never a random split, never any lookahead.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
from src.config import load_config, get_paths
cfg, paths = load_config(), get_paths()
print("as-of date:", cfg["project"]["as_of_date"])

as-of date: 2011-12-09


## 1. Model selection results

In [2]:
sel = pd.read_parquet(paths.processed / "forecast_selection.parquet")
acc = pd.read_parquet(paths.processed / "forecast_accuracy.parquet")
sel.selected_model.value_counts().to_frame("n_skus").assign(
    pct=lambda d: (100*d.n_skus/len(sel)).round(1))

,n_skus,pct
selected_model,,
moving_average,483,20.4
seasonal_naive,334,14.1
seasonal_index,274,11.6
annual_naive,259,10.9
croston,234,9.9
annual_scaled,203,8.6
ets,195,8.2
croston_sba,186,7.8
naive,180,7.6


## 2. Head-to-head — did complexity pay?\n\nModel selection is driven by error on the **30-day horizon total**, not the daily path, because a reorder quantity is built from a total, not a sequence.

In [3]:
h = acc.groupby(["is_intermittent","model"]).agg(
    n=("product_id","nunique"), horizon_abs_err=("horizon_abs_error","mean"),
    bias=("horizon_bias","mean"), mase=("mase","mean")).round(2)
print("DENSE SKUs"); display(h.loc[False].sort_values("horizon_abs_err"))
print("INTERMITTENT SKUs"); display(h.loc[True].sort_values("horizon_abs_err"))

DENSE SKUs


,n,horizon_abs_err,bias,mase
model,,,,
ets,719,174.75,-17.09,0.87
moving_average,719,184.41,-33.29,0.91
seasonal_index,719,200.14,-35.64,0.99
annual_scaled,719,266.09,68.22,1.21
seasonal_naive,719,266.93,10.54,1.08
annual_naive,719,270.42,105.31,1.23
sarima,100,380.87,-122.83,0.95
naive,719,536.82,115.34,1.35


INTERMITTENT SKUs


,n,horizon_abs_err,bias,mase
model,,,,
moving_average,1651,77.70,-18.86,1.94
seasonal_index,1651,81.11,-29.60,1.89
croston_sba,1651,83.23,-37.12,1.85
croston,1651,83.48,-32.28,1.87
seasonal_naive,1651,99.36,-9.94,2.11
annual_naive,1651,144.08,-32.68,2.07
annual_scaled,1651,145.46,-30.70,2.12
naive,1651,187.28,24.75,2.47


### SARIMA did not earn its cost

On the 100 highest-revenue dense SKUs it was **beaten by a 28-day moving
average** (381 vs 184 mean horizon error) while costing ~50x more to fit. It was
therefore not rolled out to all 2,370 SKUs. Benchmarking it on a sample and
rejecting it on evidence is the engineering decision; fitting it 7,110 times
because it sounds sophisticated is not.

## 3. Accuracy of the selected models\n\n**MASE** is the headline metric: it is defined when actuals are zero (MAPE is not) and is comparable across SKUs of very different volume. MASE < 1 beats a naive one-step forecast.

In [4]:
print(f"median MASE            : {sel.backtest_mase.median():.3f}")
print(f"SKUs beating naive     : {(sel.backtest_mase<1).sum():,} / {len(sel):,} ({100*(sel.backtest_mase<1).mean():.1f}%)")
print(f"median sMAPE           : {sel.backtest_smape.median():.1f}%   <- inflated by structural zeros")
print(f"median horizon abs err : {sel.backtest_horizon_abs_error.median():.1f} units")
sel[["backtest_mae","backtest_rmse","backtest_smape","backtest_mase"]].describe([.25,.5,.75]).round(3)

median MASE            : 0.883
SKUs beating naive     : 1,331 / 2,370 (56.2%)
median sMAPE           : 164.5%   <- inflated by structural zeros
median horizon abs err : 34.0 units


,backtest_mae,backtest_rmse,backtest_smape,backtest_mase
count,2370.000,2370.000,2370.000,2370.000
mean,7.798,15.128,158.913,1.573
std,13.225,27.285,32.872,3.281
min,0.007,0.037,40.000,0.001
25%,0.945,2.129,139.591,0.499
50%,3.255,6.403,164.492,0.883
75%,9.388,18.022,186.494,1.619
max,187.613,460.643,200.000,88.171


**On sMAPE:** the median of ~153% looks alarming and is reported anyway. With
demand zero on 75% of days, any non-zero forecast against a zero actual scores
200%. This is precisely why sMAPE is *not* the selection metric — it is shown to
demonstrate the failure mode, not to claim the forecast is bad.

## 4. The largest remaining uncertainty — and why it cannot be validated away

The as-of date sits on the Christmas peak. The 30/90-day operational window is
the post-Christmas collapse. **Every backtest fold falls in the ramp-up**, so
model selection has no evidence about post-peak behaviour.

This cannot be fixed with better validation: with only two years of history, a
seasonal model needs one year to learn the season and a second to be tested on
it — there is no third year to validate against.

So instead of silently trusting a regime-blind forecast, **both scenarios are
published**. The gap between them is the honest uncertainty.

In [5]:
fc = pd.read_parquet(paths.processed / "forecast_sku.parquet")
f90 = fc[fc.horizon_days==90]
sel_total, scen_total = f90.forecast_units.sum(), f90.forecast_units_seasonal_scenario.sum()
print(f"selected-model 90d forecast : {sel_total:>10,.0f} units")
print(f"seasonal-scenario 90d       : {scen_total:>10,.0f} units")
print(f"gap                         : {100*(sel_total/scen_total-1):>9.0f}%")

selected-model 90d forecast :  1,427,665 units
seasonal-scenario 90d       :    890,491 units
gap                         :        60%


→ Continue to `04_inventory_risk_model.ipynb`